In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import WikipediaLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)

True

In [3]:
chunk_size = 500
chunk_overlap = 50

loader = WikipediaLoader(query = "Steve Jobs", load_max_docs=5)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

docs = text_splitter.split_documents(documents=documents)

docs

[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell Wozniak

In [4]:
embeddings = HuggingFaceEmbeddings(model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL"))
vector_store = Chroma.from_documents(docs, embeddings)

vector_store

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9302.81it/s]


In [5]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x12aa8c830>, search_kwargs={'k': 5})

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert at writing a detailed explanation on the topic: '{query}'
    Create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template= template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query = query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hyde_doc = response.content
    return hyde_doc

In [9]:
query = "When was Steve Jobs fired from Apple?"

print(get_hyde_doc(query))

[SystemMessage(content="Imagine you are an expert at writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    Create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
***Disclaimer: This analysis is based on historical accounts, executive summaries, and technological biographic scholarship. The events surrounding Apple’s management changes are complex, often conflicting, and subject to varying interpretations.***

***

# The Exit, The Diagnosis, and The Return: An Analysis of Steve Jobs’ Departure from Apple

The question of when Steve Jobs was "fired" from Apple is not a matter of a simple date, but rather a nuanced historical analysis of institutional friction, managerial power struggles, and fundamental disagreements over the company's operational trajectory. To answer definitively is to oversimplify a pivotal chapter in modern business history.

However, based on the consensus of historical and technological accounts

In [10]:
matched_doc = retriever.invoke(get_hyde_doc(query))

print(matched_doc)

[SystemMessage(content="Imagine you are an expert at writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    Create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. with his early business partner Steve Wozniak as Apple Computer Company in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterward. He attended Reed College in

In [12]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_classic.document_loaders import TextLoader

doc_loader = TextLoader("../data/text_files/python_intro.txt")

docs = doc_loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='backward-compatible with earlier versions. Beginning with Python 3.5,[39] ca

In [13]:
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key="web_search"
)

hyde_embedding_function

HypotheticalDocumentEmbedder(verbose=False, base_embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), llm_chain=PromptTemplate(input_variables=['QUESTION'], input_types={}, partial_variables={}, template='Please write a passage to answer the question\nQuestion: {QUESTION}\nPassage:')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser())

In [14]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=hyde_embedding_function,
    persist_directory="output/langchain"
)

In [15]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

rag_chain = create_stuff_documents_chain(llm = llm, prompt= rag_prompt)

In [16]:
def hyde_rag_pipeline(query):
    matched_doc = vectorstore.similarity_search(query, k = 4)
    response = rag_chain.invoke({
        "input": query,
        "context": matched_doc
    })

    return response

In [17]:
query = "What are the different modules in python?"

answer = hyde_rag_pipeline(query)

print("✅ Final answer:\n", answer)

✅ Final answer:
 Based on the context provided, the text states that Python is designed to be highly extensible through modules and mentions an "extensive ('batteries-included') standard library." However, the context **does not list** the names or types of the different modules available in Python.
